In [1]:
import pandas as pd
import json

In [3]:
with open('/Users/mcargnel/Documents/mea/Big-Data-UBA-Grupo-001/TPtres/data/fraud/train_fraud_labels.json', 'r') as file:
    data = json.load(file)

In [6]:
df_target = pd.DataFrame.from_dict(data['target'], orient='index', columns=['Target']).reset_index()
cards_data = pd.read_parquet('/Users/mcargnel/Documents/mea/Big-Data-UBA-Grupo-001/TPtres/data/fraud/cards_data.parquet')
transactions_data = pd.read_parquet('/Users/mcargnel/Documents/mea/Big-Data-UBA-Grupo-001/TPtres/data/fraud/transactions_data.parquet')
users_data = pd.read_parquet('/Users/mcargnel/Documents/mea/Big-Data-UBA-Grupo-001/TPtres/data/fraud/users_data.parquet')

In [7]:
df_target = df_target.rename(columns={'index':'id'})

In [8]:
cards_data = cards_data.rename(columns={'id':'card_id'})

In [9]:
cards_data = cards_data.drop(['client_id'], axis=1)

In [10]:
users_data = users_data.rename(columns={'id':'client_id'})

In [11]:
cards_data['card_id'] = cards_data['card_id'].astype(str)
users_data['client_id'] = users_data['client_id'].astype(str)
transactions_data[['id', 'client_id', 'card_id']] = transactions_data[['id', 'client_id', 'card_id']].astype(str)

In [12]:
final_df = transactions_data.merge(df_target, on='id', how='left')

In [13]:
final_df = final_df.merge(cards_data, on='card_id', how='left')

market: US
years; 2010 - 2019

Metric,Business Question It Answers
- Confusion Matrix,(The Source of Truth) What kind of mistakes is my model making?
- Recall (Sensitivity),"Of all the real fraud, how much did we catch? (Minimizes FN)"
- Precision,"Of all the alerts we raised, how many were real? (Minimizes FP)"
- F1-Score,What is the balanced score between Precision and Recall?
- PR Curve & AUPRC,(Best Visual) How does my Precision/Recall trade-off look at all thresholds?
- MCC,(Best Single #) What is the overall correlation between my predictions and reality?

In [20]:
final_df = final_df.merge(users_data, on='client_id', how='left')

In [21]:
final_df = final_df[~final_df['Target'].isna()]

In [22]:
final_df = final_df.drop(['id', 'client_id', 'card_id', 'merchant_id', 'zip','cvv','mcc','card_number' ], axis=1)

In [23]:
final_df['Target'].value_counts()

Target
No     8901631
Yes      13332
Name: count, dtype: int64

In [24]:
13332 / (8901631 + 13332) *100

0.14954633014180765

In [25]:
final_df.to_parquet('final_fraud_df.parquet')

KeyboardInterrupt: 

In [51]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8914963 entries, 0 to 13305912
Data columns (total 29 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   date                   object 
 1   amount                 object 
 2   use_chip               object 
 3   merchant_city          object 
 4   merchant_state         object 
 5   errors                 object 
 6   Target                 object 
 7   card_brand             object 
 8   card_type              object 
 9   expires                object 
 10  has_chip               object 
 11  num_cards_issued       int64  
 12  credit_limit           object 
 13  acct_open_date         object 
 14  year_pin_last_changed  int64  
 15  card_on_dark_web       object 
 16  current_age            int64  
 17  retirement_age         int64  
 18  birth_year             int64  
 19  birth_month            int64  
 20  gender                 object 
 21  address                object 
 22  latitude              